# PREPROCESSING 

The data that we need is:

1. CERF allocations 
2. IDMC for displacements 
3. ACLED for fatalities 
4. HDX Signals 
5. EconAI 
6. Google Trends - **NEED THE COMPLETE INFORMATION FROM THE API**
7. INFORM Index

In this nootbook we will upload the data and preprocess it in order to have it in an apropiate format to start feature enginyeering. So for each dataset we have to clean the data, filter it to keep only the relevant information and produce the final files that we will use during the project. All the files should have `iso3` and `month` as columns, because these are going to be the MultiIndex of our final dataset.

In [2]:
import pandas as pd
import os
import pycountry
import numpy as np
from pathlib import Path

### CERF ALLOCATIONS

Regarding the data of the CERF allocations, we have 2 files, one with data from 2006 to 2024, and one with data of 2024 and 2025. One issue is that the formats of the data are different, so we need to create a file with data from 2006 to 2025 unifying the two formats and keeping only the relevant information. 

In [4]:
cerf_0624 = pd.read_excel("../data_raw/CERF allocations/CERF allocations 2006-Jun2024 - EconAI.xlsx", skiprows=1) # Because there's one line of text that's no need it
cerf_2425 = pd.read_excel("../data_raw/CERF allocations/CERF allocations 2024-2025.xlsx")

In [5]:
cerf_0624.head()

,Application Code,Country,Emergency Type,Application Title,Year,Amount Approved,Total Amount Required,Date of ERC Endorsement,Amount Endorsed by ERC,Geographical Areas of Implementation,...,Number of Children,Adults,Number of IDPs,Number of Returnees,Number of Refugees,Number of Host Population,Number of Other Affected People,Persons with Disabilities,2a. Overview of the humanitarian situation,2b. CERF-funded assistance
0,24-RR-BDI-65155,Burundi,Flood,Burundi RR Application May 2024 (El Nino-relat...,2024,2500773,26000000,2024-05-18,2500000,"Communes de Mutimbuzi, Kabezi, Mubimbi (Provin...",...,28142.0,31858.0,35315.0,8180.0,4478.0,12027.0,NaN,4169.0,Heavy rains induced by El Niño have caused sev...,This $2.5 million CERF allocation aims to prov...
1,24-RR-BFA-65175,Burkina Faso,Violence/Clashes,Burkina Faso RR Application May 2024 (Violence...,2024,5000007,934600000,2024-05-18,5000000,"Soum et Yagha (Sahel), Bam, Sanmatenga et Name...",...,67019.0,50681.0,NaN,81690.0,1000.0,35010.0,NaN,7002.0,Burkina Faso is facing increasing humanitarian...,"In response to the crisis, the Emergency Relie..."
2,24-RR-ZWE-64774,Zimbabwe,Drought,Zimbabwe RR Application May 2024 (El Niño-rela...,2024,3000727,429300000,2024-04-29,3000000,"Beitbridge, Binga, Bikita, Buhera, Bulilima, M...",...,85000.0,3600.0,NaN,NaN,NaN,88600.0,NaN,30.0,The El Niño-induced drought has severely exace...,This additional $3 million allocation aims to ...
3,24-RR-MWI-64768,Malawi,Drought,Malawi RR Application May 2024 (El Nino - Drou...,2024,1995676,445000000,2024-04-29,2000000,"Machinga, Nsanje districts",...,167126.0,69119.0,NaN,NaN,NaN,55000.0,181245.0,5282.0,"Prolonged dry spells, many of which lasting lo...","In response, the Emergency Relief Coordinator ..."
4,24-RR-NPL-65080,Nepal,Flood,Nepal RR Application May 2024 (Anticipatory Ac...,2024,2724993,2664091,NaT,2664091,"Sunsari, Saptari, Bardiya and Kailali",...,98479.0,185352.0,NaN,NaN,NaN,NaN,NaN,5359.0,The flat plains of the Terai in Nepal are pron...,BACKGROUND: The Emergency Relief Coordinator s...


In [6]:
len(cerf_0624)

1172

In [7]:
cerf_2425.head()

,Allocation Code,Allocation Type,Allocation Status,Allocation Source Name,Is AA Allocation,CERF Website Published Year,Continent Name,Region Name,Country Name,Emergency Types,Emergency Group for Global Reporting,Allocation Year,Total Budget Requested,Amount Approved,Total Amount Required For Response,Total Amount Received For Response,Total People Affected By Crisis,ERCEndorsementDate
0,CERF-AGO-25-RR-1468,CERF Rapid Response: Angola May 2025 (Cholera),Under Final Reporting,Rapid Response,No,2025.0,Africa,Middle Africa,Angola,Disease Outbreak - Cholera,Disease outbreak,2025.0,1800000.0,1799873.54,17000000.0,4101414.0,6045650.0,2025-05-09T00:00:00
1,CERF-BDI-24-UF-1406,CERF Underfunded Emergencies: Burundi 2024 (Po...,Under Implementation,Underfunded Emergencies,No,2024.0,Africa,Eastern Africa,Burundi,Climate / natural disaster - Flood,Climate / natural disaster,2024.0,6000000.0,5992935.38,26000000.0,12700000.0,306000.0,2024-08-15T00:00:00
2,CERF-BDI-25-RR-1461,CERF Rapid Response: Burundi Mar 2025 (Displac...,Under Final Reporting,Rapid Response,No,2025.0,Africa,Eastern Africa,Burundi,Conflict - Displacement,Conflict,2025.0,2500000.0,2499131.75,62200000.0,8000000.0,70000.0,2025-03-27T00:00:00
3,CERF-BDI-25-RR-1511,CERF Rapid Response: Burundi Dec 2025 (Refugees),Under Implementation,Rapid Response,No,2026.0,Africa,Eastern Africa,Burundi,Conflict - Refugees,Conflict,2025.0,3500000.0,3500167.98,35300000.0,3400000.0,80000.0,2025-12-29T00:00:00
4,CERF-BFA-24-UF-1410,CERF Underfunded Emergencies: Burkina Faso 202...,Under Implementation,Underfunded Emergencies,No,2024.0,Africa,Western Africa,Burkina Faso,Conflict - Violence/clashes,Conflict,2024.0,11000000.0,11000015.19,934600000.0,339600000.0,1300000.0,2024-08-15T00:00:00


In [8]:
len(cerf_2425)

126

The filters that we have to apply are the following:

1. Only keep Rapid Response.
2. Filter out Anticipatory Action Allocations.
3. Only use allocations with Emergency Types = Displacement, Human Rights or Violence/Clashes.

To do so it will depend on each file, since they don't have the same columns, buit yet the relevant information is there for both.

**Keeping only RR**

The file 0624 already contains only this allocations, so we only need to do it for the 2425 file. The Rapid Response allocations are the ones which it's Allocation Type contains Rapid Response in it:

In [9]:
cerf_2425 = cerf_2425[cerf_2425['Allocation Type'].str.contains('Rapid Response', case=False, na=False)].copy()
len(cerf_2425)

96

**Filter out Anticipatory Action**

In the first file (2006-2024) we can detect if it'a an Anticipatory Action looking at the Application Title column (it contains Anticipatory Action in the text of this column). In the case of the second file (2024-2025) there's a column `AA Allocations`that contains either a Yes or a No.

In [10]:
cerf_0624 = cerf_0624[~cerf_0624['Application Title'].str.contains('Anticipatory Action', case=False, na=False)].copy()
cerf_2425 = cerf_2425[cerf_2425['Is AA Allocation'].str.strip() != 'Yes'].copy()

In [11]:
len(cerf_0624)

1139

In [12]:
len(cerf_2425)

70

**Keep only Displacement, Human Rights or Violence/Clashes**

In the first file (2006-2024) this information is contained in the column `Emergency Type` with the names "Displacement", "Human Rights" and "Violence/Clashes". In the second one (2024-2025) `Emergency Types` under the names: "Confict - Displacement", "Conflict - Displacement, Conflict - Refugees" and "Confict - Violence/clashes".

In [13]:
condition_0624 = (
    cerf_0624['Emergency Type'].str.contains('Displacement', case=False, na=False) |
    cerf_0624['Emergency Type'].str.contains('Human Rights', case=False, na=False) |
    cerf_0624['Emergency Type'].str.contains('Violence', case=False, na=False)
)
cerf_0624 = cerf_0624[condition_0624].copy()

In [14]:
condition_2425 = (
    cerf_2425['Emergency Types'].str.contains('Displacement', case=False, na=False) |
    cerf_2425['Emergency Types'].str.contains('Violence', case=False, na=False)
)
cerf_2425 = cerf_2425[condition_2425].copy()

In [15]:
len(cerf_0624)

347

In [16]:
len(cerf_2425)

20

Now that the data is filtered, we are going to create a final csv only containing the columns that we want. We want each row to be an allocation, containing the iso3 (Country code), County Name, Allocation Date (First Project Approved Date) and Amount Approved.

All this information is in the files except the iso3 code, that we will have to manually put it.

In [17]:
# We have added "Cote d'Ivoire", "Democratic Republic of the Congo", "Venezuela Regional Refugee and Migration Crisis" and "Palestinian territory, occupied", "occupied Palestinian territory".
manual_iso3 = {
    "Brunei": "BRN",
    "East Timor": "TLS",
    "Micronesia": "FSM",
    "Bailiwick of Guernsey": "GGY",
    "Bailiwick of Jersey": "JEY",
    "Kosovo": "XKX",  # common non-ISO code
    "Russia": "RUS",
    "Vatican City": "VAT",
    "Caribbean Netherlands": "BES",
    "Curacao": "CUW",
    "Falkland Islands": "FLK",
    "Saint-Barthelemy": "BLM",
    "Saint-Martin": "MAF",
    "Sint Maarten": "SXM",
    "Palestine": "PSE",
    "occupied Palestinian territory": "PSE",
    "Palestinian territory, occupied": "PSE",
    "Turkey": "TUR",
    "Cape Verde": "CPV",
    "Democratic Republic of Congo": "COD",
    "Democratic Republic of the Congo": "COD",
    "Ivory Coast": "CIV",
    "Cote d'Ivoire": "CIV",
    "Republic of Congo": "COG",
    "Reunion": "REU",
    # no official ISO 3166-1 code for this one, keep as custom if needed
    "Akrotiri and Dhekelia": "AKD",
    "Venezuela Regional Refugee and Migration Crisis": "VEN"
}

def country_to_iso3(name):
    # manual first
    if name in manual_iso3:
        return manual_iso3[name]
    
    # then try pycountry lookup
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None


In [18]:
# ---- Apply mapping ----
cerf_0624["iso3"] = cerf_0624["Country"].apply(country_to_iso3)
cerf_2425["iso3"] = cerf_2425["Country Name"].apply(country_to_iso3)

In [19]:
cerf_0624[cerf_0624["iso3"].isna()]

,Application Code,Country,Emergency Type,Application Title,Year,Amount Approved,Total Amount Required,Date of ERC Endorsement,Amount Endorsed by ERC,Geographical Areas of Implementation,...,Adults,Number of IDPs,Number of Returnees,Number of Refugees,Number of Host Population,Number of Other Affected People,Persons with Disabilities,2a. Overview of the humanitarian situation,2b. CERF-funded assistance,iso3
206,20-RR-GLB-46341,Global,Human Rights,Global RR Application Dec 2020 (GBV programming),2021,25004109,218134542,2020-10-29,25000000,"Bangladesh, Cameroon, Colombia, Ethiopia, Iraq...",...,482674.0,279991.0,127198.0,103530.0,176692.0,79516.0,34328.0,The COVID-19 situation exacerbated already exi...,The United Nations Population Fund (UNFPA) and...,NaN


In [20]:
cerf_2425[cerf_2425["iso3"].isna()]

,Allocation Code,Allocation Type,Allocation Status,Allocation Source Name,Is AA Allocation,CERF Website Published Year,Continent Name,Region Name,Country Name,Emergency Types,Emergency Group for Global Reporting,Allocation Year,Total Budget Requested,Amount Approved,Total Amount Required For Response,Total Amount Received For Response,Total People Affected By Crisis,ERCEndorsementDate,iso3


In the file 2006-2024, the Country Name is in the column `Country`, the Allocation Date is in `Date of Earliest Project Start`, and Amount Approved in `Amount Approved`. In the other one, the Country Name is in the column `Country Name`, the Allocation Date is in `??`, and Amount Approved in `Amount Approved`.

First we are deletting all the rows that don't have the appropiate information such as what we will consider the Allocation Date and the Country Code, and finally we will joint the two files in one.

In [21]:
cerf_0624 = cerf_0624.dropna(subset=['iso3']).copy()
cerf_2425 = cerf_2425.dropna(subset=['iso3']).copy()

In [22]:
cerf_0624[cerf_0624["Date of Earliest Project Start"].isna()]

,Application Code,Country,Emergency Type,Application Title,Year,Amount Approved,Total Amount Required,Date of ERC Endorsement,Amount Endorsed by ERC,Geographical Areas of Implementation,...,Adults,Number of IDPs,Number of Returnees,Number of Refugees,Number of Host Population,Number of Other Affected People,Persons with Disabilities,2a. Overview of the humanitarian situation,2b. CERF-funded assistance,iso3
354,18-RR-COD-28607,Democratic Republic of the Congo,Displacement,DR Congo RR Application Jan 2018 (South Sudan ...,2018,0,5753307,NaT,0,Provinces de l’Ituri et du Haut Uele,...,32431.0,NaN,NaN,88976.0,NaN,NaN,NaN,NaN,NaN,COD
477,15-RR-DZA-15966,Algeria,Displacement,15-RR-DZA-15966_Algeria_Aug2015_Application,2015,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DZA
524,14-RR-SYR-12202,Syrian Arab Republic,Displacement,14-RR-SYR-12202_Syria_Oct2014_Application,2014,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SYR
642,12-RR-MLI-13485,Mali,Displacement,12-RR-MLI-13485_Mali_Jun2016_Application,2012,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MLI
649,12-RR-BDI-13271,Burundi,Displacement,12-RR-BDI-13271_Burundi_Jun2016_Application,2012,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BDI
676,12-RR-NAM-8210,Namibia,Displacement,12-RR-NAM-8210_Namibia_Jun2016_Application,2012,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NAM
687,13-RR-LBN-7018,Lebanon,Displacement,13-RR-LBN-7018_Lebanon_Jun2016_Application,2012,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LBN
721,11-RR-LBY-13475,Libya,Displacement,11-RR-LBY-13475_Libya_Jun2016_Application,2011,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LBY
725,11-RR-LBY-13068,Libya,Displacement,11-RR-LBY-13068_Libya_Jun2016_Application,2011,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LBY
726,11-RR-LBY-13064,Libya,Displacement,11-RR-LBY-13064_Libya_Jun2016_Application,2011,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LBY


In [23]:
cerf_0624 = cerf_0624.dropna(subset=['Date of Earliest Project Start']).copy()
len(cerf_0624)

331

In [24]:
cerf_0624["Country Name"] = cerf_0624["Country"]
cerf_0624["Allocation Date"] = cerf_0624["Date of Earliest Project Start"]

In [25]:
cerf_2425[cerf_2425["ERCEndorsementDate"].isna()]

,Allocation Code,Allocation Type,Allocation Status,Allocation Source Name,Is AA Allocation,CERF Website Published Year,Continent Name,Region Name,Country Name,Emergency Types,Emergency Group for Global Reporting,Allocation Year,Total Budget Requested,Amount Approved,Total Amount Required For Response,Total Amount Received For Response,Total People Affected By Crisis,ERCEndorsementDate,iso3


In [26]:
cerf_2425 = cerf_2425.dropna(subset=['ERCEndorsementDate']).copy()
len(cerf_2425)

20

In [27]:
cerf_2425["Allocation Date"] = cerf_2425["ERCEndorsementDate"]

In [28]:
cols_to_keep = ["iso3", "Country Name", "Allocation Date", "Amount Approved"]

df1_clean = cerf_0624[cols_to_keep]
df2_clean = cerf_2425[cols_to_keep]

cerf_clean = pd.concat([df1_clean, df2_clean], ignore_index=True)

cerf_clean.to_csv("../data_clean/cerf_clean.csv", index=False)

### IDMC

The IDMC data consist of one file with the wrangled data.

In [29]:
idmc = pd.read_csv("../data_raw/IDCM/idmc_conflict_wrangled_20260424.csv", sep=",")
idmc.head()

,iso3,displacement_type,date,displacement_daily,displacement_7d,displacement_30d
0,AB9,Conflict,1/1/2018,0.0,NaN,NaN
1,AB9,Conflict,1/2/2018,0.0,NaN,NaN
2,AB9,Conflict,1/3/2018,0.0,NaN,NaN
3,AB9,Conflict,1/4/2018,0.0,NaN,NaN
4,AB9,Conflict,1/5/2018,0.0,NaN,NaN


First we filter so that we only keep the ones where the `displacement_type`= "Conflict". Then we create the variable `month` from the `date` and we group by `iso3` and `month`. We also delete all the observations that doesen't have `date` or `iso3`.

In [30]:
idmc = idmc[idmc["displacement_type"].astype(str).str.strip() == "Conflict"].copy()

# Convert date to datetime
idmc["date"] = pd.to_datetime(
    idmc["date"],
    format="%m/%d/%Y",  # e.g. 01/01/2018
    errors="coerce"
)
idmc = idmc.dropna(subset=["date"])
idmc = idmc.dropna(subset=["iso3"])

# Extract year-month
idmc["month"] = idmc["date"].dt.to_period("M").dt.to_timestamp()

# Aggregate monthly displacement (conflict only)
idmc = (
    idmc
    .groupby(["iso3", "month"])["displacement_daily"]
    .sum()
    .reset_index()
    .rename(columns={"displacement_daily": "monthly_displacement"})
)

idmc.to_csv("../data_clean/idmc_clean.csv", index=False)

In [31]:
idmc.head()

,iso3,month,monthly_displacement
0,AB9,2018-01-01,0.0
1,AB9,2018-02-01,0.0
2,AB9,2018-03-01,0.0
3,AB9,2018-04-01,0.0
4,AB9,2018-05-01,0.0


### ACLED

The ACLED data that we have are 6 files, each one corresponding to the data of one region. In this case the preprocessing consist on manually aggregating de 6 files, and create and keep only the columns that might be relevant for our study.

In [32]:
acled_africa = pd.read_excel("../data_raw/ACLED/Africa_aggregated_data_up_to_week_of-2026-03-21.xlsx") 
acled_asia = pd.read_excel("../data_raw/ACLED/Asia-Pacific_aggregated_data_up_to_week_of-2026-03-28.xlsx") 
acled_europe = pd.read_excel("../data_raw/ACLED/Europe-Central-Asia_aggregated_data_up_to_week_of-2026-03-28.xlsx") 
acled_latin = pd.read_excel("../data_raw/ACLED/Latin-America-the-Caribbean_aggregated_data_up_to_week_of-2026-03-21.xlsx") 
acled_middle = pd.read_excel("../data_raw/ACLED/Middle-East_aggregated_data_up_to_week_of-2026-03-21.xlsx") 
acled_us = pd.read_excel("../data_raw/ACLED/US-and-Canada_aggregated_data_up_to_week_of-2026-03-28.xlsx") 

In [33]:
print(f"Number of rows in Africa: {len(acled_africa)}")
print(f"Number of rows in Asia-Pacific: {len(acled_asia)}")
print(f"Number of rows in Europe & Central Asia: {len(acled_europe)}")
print(f"Number of rows in Latin America & Caribbean: {len(acled_latin)}")
print(f"Number of rows in Middle East: {len(acled_middle)}")
print(f"Number of rows in US and Canada: {len(acled_us)}")

Number of rows in Africa: 268511
Number of rows in Asia-Pacific: 208289
Number of rows in Europe & Central Asia: 118846
Number of rows in Latin America & Caribbean: 171517
Number of rows in Middle East: 145353
Number of rows in US and Canada: 22478


Let's now check what this file contains so that we can choose which columns we want to keep and how.

In [34]:
acled_africa.head()

,WEEK,REGION,COUNTRY,ADMIN1,EVENT_TYPE,SUB_EVENT_TYPE,EVENTS,FATALITIES,POPULATION_EXPOSURE,DISORDER_TYPE,ID,CENTROID_LATITUDE,CENTROID_LONGITUDE
0,2004-10-23,Northern Africa,Algeria,Adrar,Battles,Armed clash,1,2,NaN,Political violence,47.0,26.4839,-1.388
1,2005-04-23,Northern Africa,Algeria,Adrar,Battles,Armed clash,1,0,NaN,Political violence,47.0,26.4839,-1.388
2,2005-06-25,Northern Africa,Algeria,Adrar,Battles,Armed clash,1,14,NaN,Political violence,47.0,26.4839,-1.388
3,2008-12-13,Northern Africa,Algeria,Adrar,Battles,Armed clash,1,3,NaN,Political violence,47.0,26.4839,-1.388
4,2009-04-18,Northern Africa,Algeria,Adrar,Battles,Armed clash,1,2,NaN,Political violence,47.0,26.4839,-1.388


The columns that we have to create are `iso3` and `month`. We are dropping all the rows that doesn't have some of these two values, since they are important for our analisis.  

In [35]:
# We have added "Cote d'Ivoire", "Democratic Republic of the Congo", "Venezuela Regional Refugee and Migration Crisis" and "Palestinian territory, occupied", "occupied Palestinian territory".
manual_iso3 = {
    "Brunei": "BRN",
    "East Timor": "TLS",
    "Micronesia": "FSM",
    "Bailiwick of Guernsey": "GGY",
    "Bailiwick of Jersey": "JEY",
    "Kosovo": "XKX",  # common non-ISO code
    "Russia": "RUS",
    "Vatican City": "VAT",
    "Caribbean Netherlands": "BES",
    "Curacao": "CUW",
    "Falkland Islands": "FLK",
    "Saint-Barthelemy": "BLM",
    "Saint-Martin": "MAF",
    "Sint Maarten": "SXM",
    "Palestine": "PSE",
    "occupied Palestinian territory": "PSE",
    "Palestinian territory, occupied": "PSE",
    "Turkey": "TUR",
    "Cape Verde": "CPV",
    "Democratic Republic of Congo": "COD",
    "Democratic Republic of the Congo": "COD",
    "Ivory Coast": "CIV",
    "Cote d'Ivoire": "CIV",
    "Republic of Congo": "COG",
    "Reunion": "REU",
    # no official ISO 3166-1 code for this one, keep as custom if needed
    "Akrotiri and Dhekelia": "AKD",
    "Venezuela Regional Refugee and Migration Crisis": "VEN"
}

def country_to_iso3(name):
    # manual first
    if name in manual_iso3:
        return manual_iso3[name]
    
    # then try pycountry lookup
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None


In [36]:
acled_africa["iso3"] = acled_africa["COUNTRY"].apply(country_to_iso3)
acled_asia["iso3"] = acled_asia["COUNTRY"].apply(country_to_iso3)
acled_europe["iso3"] = acled_europe["COUNTRY"].apply(country_to_iso3)
acled_latin["iso3"] = acled_latin["COUNTRY"].apply(country_to_iso3)
acled_middle["iso3"] = acled_middle["COUNTRY"].apply(country_to_iso3)
acled_us["iso3"] = acled_us["COUNTRY"].apply(country_to_iso3)

Check if there are countries missing ISO3:

In [37]:
missing_africa = acled_africa[acled_africa["iso3"].isna()]["COUNTRY"].unique()
print(f"Africa missing: {missing_africa}")

missing_asia = acled_asia[acled_asia["iso3"].isna()]["COUNTRY"].unique()
print(f"Asia-Pacific missing: {missing_asia}")

missing_europe = acled_europe[acled_europe["iso3"].isna()]["COUNTRY"].unique()
print(f"Europe & Central Asia missing: {missing_europe}")

missing_latin = acled_latin[acled_latin["iso3"].isna()]["COUNTRY"].unique()
print(f"Latin America missing: {missing_latin}")

missing_middle = acled_middle[acled_middle["iso3"].isna()]["COUNTRY"].unique()
print(f"Middle East missing: {missing_middle}")

missing_us = acled_us[acled_us["iso3"].isna()]["COUNTRY"].unique()
print(f"US and Canada missing: {missing_us}")

Africa missing: <StringArray>
[]
Length: 0, dtype: str
Asia-Pacific missing: <StringArray>
[]
Length: 0, dtype: str
Europe & Central Asia missing: <StringArray>
[]
Length: 0, dtype: str
Latin America missing: <StringArray>
[]
Length: 0, dtype: str
Middle East missing: <StringArray>
[]
Length: 0, dtype: str
US and Canada missing: <StringArray>
[]
Length: 0, dtype: str


In [38]:
acled_africa = acled_africa.dropna(subset=["iso3"])
acled_asia = acled_asia.dropna(subset=["iso3"])
acled_europe = acled_europe.dropna(subset=["iso3"])
acled_latin = acled_latin.dropna(subset=["iso3"])
acled_middle = acled_middle.dropna(subset=["iso3"])
acled_us = acled_us.dropna(subset=["iso3"])

In [39]:
acled_africa["FATALITIES"] = pd.to_numeric(acled_africa["FATALITIES"], errors="coerce").fillna(0)
acled_africa["week"] = pd.to_datetime(acled_africa["WEEK"], errors="coerce")
acled_africa = acled_africa.dropna(subset=["week"])
acled_africa["month"] = acled_africa["week"].dt.to_period("M").dt.to_timestamp()

acled_asia["FATALITIES"] = pd.to_numeric(acled_asia["FATALITIES"], errors="coerce").fillna(0)
acled_asia["week"] = pd.to_datetime(acled_asia["WEEK"], errors="coerce")
acled_asia = acled_asia.dropna(subset=["week"])
acled_asia["month"] = acled_asia["week"].dt.to_period("M").dt.to_timestamp()

acled_europe["FATALITIES"] = pd.to_numeric(acled_europe["FATALITIES"], errors="coerce").fillna(0)
acled_europe["week"] = pd.to_datetime(acled_europe["WEEK"], errors="coerce")
acled_europe = acled_europe.dropna(subset=["week"])
acled_europe["month"] = acled_europe["week"].dt.to_period("M").dt.to_timestamp()

acled_latin["FATALITIES"] = pd.to_numeric(acled_latin["FATALITIES"], errors="coerce").fillna(0)
acled_latin["week"] = pd.to_datetime(acled_latin["WEEK"], errors="coerce")
acled_latin = acled_latin.dropna(subset=["week"])
acled_latin["month"] = acled_latin["week"].dt.to_period("M").dt.to_timestamp()

acled_middle["FATALITIES"] = pd.to_numeric(acled_middle["FATALITIES"], errors="coerce").fillna(0)
acled_middle["week"] = pd.to_datetime(acled_middle["WEEK"], errors="coerce")
acled_middle = acled_middle.dropna(subset=["week"])
acled_middle["month"] = acled_middle["week"].dt.to_period("M").dt.to_timestamp()

acled_us["FATALITIES"] = pd.to_numeric(acled_us["FATALITIES"], errors="coerce").fillna(0)
acled_us["week"] = pd.to_datetime(acled_us["WEEK"], errors="coerce")
acled_us = acled_us.dropna(subset=["week"])         
acled_us["month"] = acled_us["week"].dt.to_period("M").dt.to_timestamp()

In [40]:
acled_africa.head()

,WEEK,REGION,COUNTRY,ADMIN1,EVENT_TYPE,SUB_EVENT_TYPE,EVENTS,FATALITIES,POPULATION_EXPOSURE,DISORDER_TYPE,ID,CENTROID_LATITUDE,CENTROID_LONGITUDE,iso3,week,month
0,2004-10-23,Northern Africa,Algeria,Adrar,Battles,Armed clash,1,2,NaN,Political violence,47.0,26.4839,-1.388,DZA,2004-10-23,2004-10-01
1,2005-04-23,Northern Africa,Algeria,Adrar,Battles,Armed clash,1,0,NaN,Political violence,47.0,26.4839,-1.388,DZA,2005-04-23,2005-04-01
2,2005-06-25,Northern Africa,Algeria,Adrar,Battles,Armed clash,1,14,NaN,Political violence,47.0,26.4839,-1.388,DZA,2005-06-25,2005-06-01
3,2008-12-13,Northern Africa,Algeria,Adrar,Battles,Armed clash,1,3,NaN,Political violence,47.0,26.4839,-1.388,DZA,2008-12-13,2008-12-01
4,2009-04-18,Northern Africa,Algeria,Adrar,Battles,Armed clash,1,2,NaN,Political violence,47.0,26.4839,-1.388,DZA,2009-04-18,2009-04-01


In [41]:
print(f"Number of rows in Africa: {len(acled_africa)}")
print(f"Number of rows in Asia-Pacific: {len(acled_asia)}")
print(f"Number of rows in Europe & Central Asia: {len(acled_europe)}")
print(f"Number of rows in Latin America & Caribbean: {len(acled_latin)}")
print(f"Number of rows in Middle East: {len(acled_middle)}")
print(f"Number of rows in US and Canada: {len(acled_us)}")

Number of rows in Africa: 268511
Number of rows in Asia-Pacific: 208289
Number of rows in Europe & Central Asia: 118846
Number of rows in Latin America & Caribbean: 171517
Number of rows in Middle East: 145353
Number of rows in US and Canada: 22478


Finally, before aggregating the 6 files, for each of them we want to group the data by `iso3` x `month`, and keep the rows `EVENT_TYPE`, `SUB_EVENT_TYPE`, `EVENTS`, `FATALITIES` and `DISORDER_TYPE`.

The `EVENT_TYPE`, `SUB_EVENT_TYPE` and `DISORDER_TYPE` columns will contain a list of all the diferent types that week. `EVENTS` and `FATALITIES` will contain the sum of events and fatalities over the week. 

In [42]:
acled_africa_grouped = acled_africa.groupby(['iso3', 'month']).agg({
    'EVENT_TYPE': lambda x: list(x.unique()),
    'SUB_EVENT_TYPE': lambda x: list(x.unique()),
    'DISORDER_TYPE': lambda x: list(x.unique()),
    'EVENTS': 'sum',
    'FATALITIES': 'sum'
}).reset_index()

acled_asia_grouped = acled_asia.groupby(['iso3', 'month']).agg({
    'EVENT_TYPE': lambda x: list(x.unique()),
    'SUB_EVENT_TYPE': lambda x: list(x.unique()),
    'DISORDER_TYPE': lambda x: list(x.unique()),
    'EVENTS': 'sum',
    'FATALITIES': 'sum'
}).reset_index()

acled_europe_grouped = acled_europe.groupby(['iso3', 'month']).agg({
    'EVENT_TYPE': lambda x: list(x.unique()),
    'SUB_EVENT_TYPE': lambda x: list(x.unique()),
    'DISORDER_TYPE': lambda x: list(x.unique()), 
    'EVENTS': 'sum',
    'FATALITIES': 'sum'
}).reset_index()   

acled_latin_grouped = acled_latin.groupby(['iso3', 'month']).agg({
    'EVENT_TYPE': lambda x: list(x.unique()),
    'SUB_EVENT_TYPE': lambda x: list(x.unique()),
    'DISORDER_TYPE': lambda x: list(x.unique()),
    'EVENTS': 'sum',
    'FATALITIES': 'sum'
}).reset_index()

acled_middle_grouped = acled_middle.groupby(['iso3', 'month']).agg({
    'EVENT_TYPE': lambda x: list(x.unique()),
    'SUB_EVENT_TYPE': lambda x: list(x.unique()),
    'DISORDER_TYPE': lambda x: list(x.unique()),
    'EVENTS': 'sum',
    'FATALITIES': 'sum'
}).reset_index()

acled_us_grouped = acled_us.groupby(['iso3', 'month']).agg({
    'EVENT_TYPE': lambda x: list(x.unique()),
    'SUB_EVENT_TYPE': lambda x: list(x.unique()),
    'DISORDER_TYPE': lambda x: list(x.unique()),
    'EVENTS': 'sum',
    'FATALITIES': 'sum'
}).reset_index()

In [43]:
acled_africa_grouped.head()

,iso3,month,EVENT_TYPE,SUB_EVENT_TYPE,DISORDER_TYPE,EVENTS,FATALITIES
0,AGO,1996-12-01,[Violence against civilians],[Attack],[Political violence],2,40
1,AGO,1997-01-01,"[Strategic developments, Violence against civi...","[Looting/property destruction, Attack]","[Strategic developments, Political violence]",2,0
2,AGO,1997-02-01,"[Battles, Violence against civilians, Strategi...","[Armed clash, Attack, Government regains terri...","[Political violence, Strategic developments, D...",16,1027
3,AGO,1997-03-01,"[Battles, Violence against civilians]","[Armed clash, Attack]",[Political violence],5,63
4,AGO,1997-04-01,"[Battles, Protests, Violence against civilians]","[Government regains territory, Peaceful protes...","[Political violence, Demonstrations]",15,200


In [44]:
print(f"Number of rows in Africa: {len(acled_africa_grouped)}")
print(f"Number of rows in Asia-Pacific: {len(acled_asia_grouped)}")
print(f"Number of rows in Europe & Central Asia: {len(acled_europe_grouped)}")
print(f"Number of rows in Latin America & Caribbean: {len(acled_latin_grouped)}")
print(f"Number of rows in Middle East: {len(acled_middle_grouped)}")
print(f"Number of rows in US and Canada: {len(acled_us_grouped)}")

Number of rows in Africa: 12527
Number of rows in Asia-Pacific: 3380
Number of rows in Europe & Central Asia: 4303
Number of rows in Latin America & Caribbean: 3349
Number of rows in Middle East: 1508
Number of rows in US and Canada: 169


We concatenate the 6 files:

In [45]:
all_regions = [
    acled_africa_grouped, 
    acled_asia_grouped, 
    acled_europe_grouped, 
    acled_latin_grouped,
    acled_middle_grouped,  
    acled_us_grouped
]

acled_final = pd.concat(all_regions, ignore_index=True)
acled_final = acled_final.sort_values(by=['month', 'iso3'])
acled_final = acled_final.rename(columns={"EVENT_TYPE": "event_type"})
acled_final = acled_final.rename(columns={"SUB_EVENT_TYPE": "sub_event_type"})
acled_final = acled_final.rename(columns={"DISORDER_TYPE": "disorder_type"})
acled_final = acled_final.rename(columns={"EVENTS": "events"})
acled_final = acled_final.rename(columns={"FATALITIES": "fatalities"})
acled_final.to_csv("../data_clean/acled_clean.csv", index=False)

print(f"Number of rows of the final file: {len(acled_final)}.")
print(f"Included columns: {list(acled_final.columns)}")

Number of rows of the final file: 25236.
Included columns: ['iso3', 'month', 'event_type', 'sub_event_type', 'disorder_type', 'events', 'fatalities']


### HDX Signals

Regarding the data of HDX Signals, we have 1 file containing all the relevant information.

In [46]:
hdx = pd.read_csv("../data_raw/HDX Signals/hdx_signals.csv", sep=",")

In [47]:
hdx.head()

,iso3,location,region,hrp_location,indicator_id,date,alert_level,value,plot,map,...,summary_long,summary_short,summary_source,hdx_url,source_url,other_urls,further_information,campaign_url,campaign_date,signals_version
0,ARM,Armenia,Europe,False,wfp_market_monitor,2021-07-01 00:00:00,High concern,28.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,NaN,28% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfEA#ARM,2021-07-01,0.1.0
1,BDI,Burundi,Southern and Eastern Africa,False,wfp_market_monitor,2020-11-01 00:00:00,Medium concern,10.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,NaN,10% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfDc#BDI,2020-11-01,0.1.0
2,BDI,Burundi,Southern and Eastern Africa,False,wfp_market_monitor,2022-05-01 00:00:00,Medium concern,11.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,NaN,11% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfGA#BDI,2022-05-01,0.1.0
3,BDI,Burundi,Southern and Eastern Africa,False,wfp_market_monitor,2023-01-01 00:00:00,High concern,39.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,NaN,39% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfHM#BDI,2023-01-01,0.1.0
4,BEN,Benin,West and Central Africa,False,wfp_market_monitor,2020-11-01 00:00:00,Medium concern,21.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,NaN,21% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfDc#BEN,2020-11-01,0.1.0


First we want to check that all the observations have `iso3`and `date`, as both are imprecindible for the study. We will drop the observations without them.

In [48]:
hdx["date"] = pd.to_datetime(hdx["date"], errors="coerce")
hdx[hdx["date"].isna()]

,iso3,location,region,hrp_location,indicator_id,date,alert_level,value,plot,map,...,summary_long,summary_short,summary_source,hdx_url,source_url,other_urls,further_information,campaign_url,campaign_date,signals_version


In [49]:
hdx = hdx.dropna(subset=["date"]).copy()

In [50]:
hdx[hdx["iso3"].isna()]

,iso3,location,region,hrp_location,indicator_id,date,alert_level,value,plot,map,...,summary_long,summary_short,summary_source,hdx_url,source_url,other_urls,further_information,campaign_url,campaign_date,signals_version


In [51]:
hdx = hdx.dropna(subset=["iso3"]).copy()

Now we create the column `month` from `date`:

In [52]:
hdx["month"] = hdx["date"].dt.to_period("M").dt.to_timestamp()

In [53]:
hdx.head()

,iso3,location,region,hrp_location,indicator_id,date,alert_level,value,plot,map,...,summary_short,summary_source,hdx_url,source_url,other_urls,further_information,campaign_url,campaign_date,signals_version,month
0,ARM,Armenia,Europe,False,wfp_market_monitor,2021-07-01,High concern,28.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,28% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfEA#ARM,2021-07-01,0.1.0,2021-07-01
1,BDI,Burundi,Southern and Eastern Africa,False,wfp_market_monitor,2020-11-01,Medium concern,10.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,10% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfDc#BDI,2020-11-01,0.1.0,2020-11-01
2,BDI,Burundi,Southern and Eastern Africa,False,wfp_market_monitor,2022-05-01,Medium concern,11.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,11% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfGA#BDI,2022-05-01,0.1.0,2022-05-01
3,BDI,Burundi,Southern and Eastern Africa,False,wfp_market_monitor,2023-01-01,High concern,39.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,39% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfHM#BDI,2023-01-01,0.1.0,2023-01-01
4,BEN,Benin,West and Central Africa,False,wfp_market_monitor,2020-11-01,Medium concern,21.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,21% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfDc#BEN,2020-11-01,0.1.0,2020-11-01


In [54]:
hdx = hdx.rename(columns={"alert_level": "hdx_alert_level"})
hdx = hdx.rename(columns={"value": "hdx_value"})

Finally we group all the data for `iso3`x `month`, and we keep only the relevant columns.

In [55]:
hdx_grouped = hdx.groupby(['iso3', 'month']).agg({
    'hdx_alert_level': lambda x: list(x.unique()),
    'hdx_value': 'sum', # Maybe better to take the max?? Or have all of them separately??
}).reset_index()

hdx_grouped.to_csv("../data_clean/hdx_clean.csv", index=False)


### EconAI

In [56]:
risk_3 = pd.read_csv("../data_raw/EconAI/conflictforecast_ons_armedconf_03.csv", low_memory=False)
logfat_3 = pd.read_csv("../data_raw/EconAI/conflictforecast_int_lnbest_03.csv", low_memory=False)
risk_12 = pd.read_csv("../data_raw/EconAI/conflictforecast_ons_armedconf_12.csv", low_memory=False)
logfat_12 = pd.read_csv("../data_raw/EconAI/conflictforecast_int_lnbest_12.csv", low_memory=False)

Let's first preprocess the files containing the risk prediction of the next 3 and 12 months.

In [57]:
risk_3.head()

,isocode,period,ons_armedconf_03_target,ons_armedconf_03_text,ons_armedconf_03_hist,ons_armedconf_03_all,ons_armedconf_03_naive,fatalities_UCDP,armedconf,discounted_anyviolence,...,stock_topic_14,stock_topic_2,stock_topic_3,stock_topic_4,stock_topic_5,stock_topic_6,stock_topic_7,stock_topic_8,stock_topic_9,tokens
0,AFG,201001,NaN,0.269263,0.998556,0.983878,1.0,344.0,1,0.999995,...,0.031562,0.147948,0.010954,0.002365,0.039283,0.041238,0.022845,0.002615,0.013236,112530.0
1,AFG,201002,NaN,0.257331,0.998499,0.991798,1.0,536.0,1,0.999995,...,0.033501,0.146538,0.011173,0.002334,0.038820,0.038723,0.022851,0.002433,0.012932,91058.0
2,AFG,201003,NaN,0.270456,0.998045,0.995489,1.0,407.0,1,0.999995,...,0.032701,0.138953,0.011799,0.002804,0.041742,0.036860,0.023159,0.002396,0.013774,102355.0
3,AFG,201004,NaN,0.261449,0.999598,0.999792,1.0,503.0,1,0.999995,...,0.034657,0.134142,0.012519,0.003110,0.043693,0.035465,0.024217,0.002324,0.014460,67243.0
4,AFG,201005,NaN,0.276905,0.995243,0.998218,1.0,502.0,1,0.999996,...,0.036816,0.131677,0.012806,0.002888,0.042615,0.034363,0.024350,0.002201,0.014451,76618.0


In [58]:
risk_12.head()

,isocode,period,ons_armedconf_12_target,ons_armedconf_12_text,ons_armedconf_12_hist,ons_armedconf_12_all,ons_armedconf_12_naive,fatalities_UCDP,armedconf,discounted_anyviolence,...,stock_topic_14,stock_topic_2,stock_topic_3,stock_topic_4,stock_topic_5,stock_topic_6,stock_topic_7,stock_topic_8,stock_topic_9,tokens
0,AFG,201001,NaN,0.534497,0.979979,0.987250,1.0,344.0,1,0.999995,...,0.031486,0.145819,0.010593,0.002425,0.039774,0.041119,0.023396,0.002611,0.013429,112530.0
1,AFG,201002,NaN,0.528127,0.982736,0.988044,1.0,536.0,1,0.999995,...,0.033440,0.144435,0.010810,0.002392,0.039358,0.038659,0.023398,0.002422,0.013095,91058.0
2,AFG,201003,NaN,0.533111,0.984456,0.989177,1.0,407.0,1,0.999995,...,0.032664,0.136855,0.011416,0.002888,0.042325,0.036741,0.023718,0.002390,0.013967,102355.0
3,AFG,201004,NaN,0.528683,0.986487,0.992189,1.0,503.0,1,0.999995,...,0.034651,0.132079,0.012109,0.003186,0.044280,0.035352,0.024767,0.002314,0.014655,67243.0
4,AFG,201005,NaN,0.533341,0.988304,0.992714,1.0,502.0,1,0.999996,...,0.036827,0.129605,0.012372,0.002955,0.043213,0.034278,0.024853,0.002194,0.014622,76618.0


In [59]:
risk_3 = risk_3.rename(columns={"isocode": "iso3"})
risk_3 = risk_3.rename(columns={"ons_armedconf_03_all": "risk_3"})
risk_3 = risk_3.rename(columns={"period": "month"})

# Convert period to datetime 
risk_3["month"] = pd.to_datetime(risk_3["month"], format="%Y%m")

risk_3.head()

,iso3,month,ons_armedconf_03_target,ons_armedconf_03_text,ons_armedconf_03_hist,risk_3,ons_armedconf_03_naive,fatalities_UCDP,armedconf,discounted_anyviolence,...,stock_topic_14,stock_topic_2,stock_topic_3,stock_topic_4,stock_topic_5,stock_topic_6,stock_topic_7,stock_topic_8,stock_topic_9,tokens
0,AFG,2010-01-01,NaN,0.269263,0.998556,0.983878,1.0,344.0,1,0.999995,...,0.031562,0.147948,0.010954,0.002365,0.039283,0.041238,0.022845,0.002615,0.013236,112530.0
1,AFG,2010-02-01,NaN,0.257331,0.998499,0.991798,1.0,536.0,1,0.999995,...,0.033501,0.146538,0.011173,0.002334,0.038820,0.038723,0.022851,0.002433,0.012932,91058.0
2,AFG,2010-03-01,NaN,0.270456,0.998045,0.995489,1.0,407.0,1,0.999995,...,0.032701,0.138953,0.011799,0.002804,0.041742,0.036860,0.023159,0.002396,0.013774,102355.0
3,AFG,2010-04-01,NaN,0.261449,0.999598,0.999792,1.0,503.0,1,0.999995,...,0.034657,0.134142,0.012519,0.003110,0.043693,0.035465,0.024217,0.002324,0.014460,67243.0
4,AFG,2010-05-01,NaN,0.276905,0.995243,0.998218,1.0,502.0,1,0.999996,...,0.036816,0.131677,0.012806,0.002888,0.042615,0.034363,0.024350,0.002201,0.014451,76618.0


In [60]:
risk_12 = risk_12.rename(columns={"isocode": "iso3"})
risk_12 = risk_12.rename(columns={"ons_armedconf_12_all": "risk_12"})
risk_12 = risk_12.rename(columns={"period": "month"})

# Convert period to datetime 
risk_12["month"] = pd.to_datetime(risk_12["month"], format="%Y%m")

risk_12.head()

,iso3,month,ons_armedconf_12_target,ons_armedconf_12_text,ons_armedconf_12_hist,risk_12,ons_armedconf_12_naive,fatalities_UCDP,armedconf,discounted_anyviolence,...,stock_topic_14,stock_topic_2,stock_topic_3,stock_topic_4,stock_topic_5,stock_topic_6,stock_topic_7,stock_topic_8,stock_topic_9,tokens
0,AFG,2010-01-01,NaN,0.534497,0.979979,0.987250,1.0,344.0,1,0.999995,...,0.031486,0.145819,0.010593,0.002425,0.039774,0.041119,0.023396,0.002611,0.013429,112530.0
1,AFG,2010-02-01,NaN,0.528127,0.982736,0.988044,1.0,536.0,1,0.999995,...,0.033440,0.144435,0.010810,0.002392,0.039358,0.038659,0.023398,0.002422,0.013095,91058.0
2,AFG,2010-03-01,NaN,0.533111,0.984456,0.989177,1.0,407.0,1,0.999995,...,0.032664,0.136855,0.011416,0.002888,0.042325,0.036741,0.023718,0.002390,0.013967,102355.0
3,AFG,2010-04-01,NaN,0.528683,0.986487,0.992189,1.0,503.0,1,0.999995,...,0.034651,0.132079,0.012109,0.003186,0.044280,0.035352,0.024767,0.002314,0.014655,67243.0
4,AFG,2010-05-01,NaN,0.533341,0.988304,0.992714,1.0,502.0,1,0.999996,...,0.036827,0.129605,0.012372,0.002955,0.043213,0.034278,0.024853,0.002194,0.014622,76618.0


Keep only the rows that we want (`iso3`, `period` and risk columns):

In [61]:
# Sort
risk_3 = risk_3.sort_values(["iso3", "month"])
risk_12 = risk_12.sort_values(["iso3", "month"])

cols_keep_3 = [
    "iso3",
    "month",
    "risk_3"
]

cols_to_keep_12 = [
    "iso3",
    "month",
    "risk_12"
]

risk_3 = risk_3[cols_keep_3].copy()
risk_12 = risk_12[cols_to_keep_12].copy()

Delete the rows that have missing values.

In [ ]:
risk_3 = risk_3.dropna().reset_index(drop=True)
risk_12 = risk_12.dropna().reset_index(drop=True)

Now check the files logfat, containing the prediction of the number of fatalities of a country in the next 3 and 12 months.

In [63]:
logfat_3.head()

,isocode,period,int_lnbest_03_target,int_lnbest_03_text,int_lnbest_03_hist,int_lnbest_03_all,int_lnbest_03_naive,fatalities_UCDP,lnbest,discounted_anyviolence,...,stock_topic_14,stock_topic_2,stock_topic_3,stock_topic_4,stock_topic_5,stock_topic_6,stock_topic_7,stock_topic_8,stock_topic_9,tokens
0,AFG,201001,7.277248,6.357100,6.830939,6.844159,7.120444,344.0,5.843544,0.999995,...,0.031562,0.147948,0.010954,0.002365,0.039283,0.041238,0.022845,0.002615,0.013236,112530.0
1,AFG,201002,7.253470,6.358596,6.857611,6.883800,7.127694,536.0,6.285998,0.999995,...,0.033501,0.146538,0.011173,0.002334,0.038820,0.038723,0.022851,0.002433,0.012932,91058.0
2,AFG,201003,7.604396,6.466978,6.855770,6.898818,7.160846,407.0,6.011267,0.999995,...,0.032701,0.138953,0.011799,0.002804,0.041742,0.036860,0.023159,0.002396,0.013774,102355.0
3,AFG,201004,7.720905,6.363206,6.891226,6.939510,7.277248,503.0,6.222576,0.999995,...,0.034657,0.134142,0.012519,0.003110,0.043693,0.035465,0.024217,0.002324,0.014460,67243.0
4,AFG,201005,7.876638,6.494570,6.917318,6.938101,7.253470,502.0,6.220590,0.999996,...,0.036816,0.131677,0.012806,0.002888,0.042615,0.034363,0.024350,0.002201,0.014451,76618.0


In [64]:
logfat_12.head()

,isocode,period,int_lnbest_12_target,int_lnbest_12_text,int_lnbest_12_hist,int_lnbest_12_all,int_lnbest_12_naive,fatalities_UCDP,lnbest,discounted_anyviolence,...,stock_topic_14,stock_topic_2,stock_topic_3,stock_topic_4,stock_topic_5,stock_topic_6,stock_topic_7,stock_topic_8,stock_topic_9,tokens
0,AFG,201001,8.900004,6.612788,8.104944,8.102876,8.786609,344.0,5.843544,0.999995,...,0.031486,0.145819,0.010593,0.002425,0.039774,0.041119,0.023396,0.002611,0.013429,112530.0
1,AFG,201002,8.888205,6.433330,8.170137,8.129857,8.818186,536.0,6.285998,0.999995,...,0.033440,0.144435,0.010810,0.002392,0.039358,0.038659,0.023398,0.002422,0.013095,91058.0
2,AFG,201003,8.893847,6.608980,8.138193,8.131346,8.811503,407.0,6.011267,0.999995,...,0.032664,0.136855,0.011416,0.002888,0.042325,0.036741,0.023718,0.002390,0.013967,102355.0
3,AFG,201004,8.900685,6.563129,8.167914,8.149257,8.820847,503.0,6.222576,0.999995,...,0.034651,0.132079,0.012109,0.003186,0.044280,0.035352,0.024767,0.002314,0.014655,67243.0
4,AFG,201005,8.931420,6.572965,8.156089,8.141677,8.773694,502.0,6.220590,0.999996,...,0.036827,0.129605,0.012372,0.002955,0.043213,0.034278,0.024853,0.002194,0.014622,76618.0


In [65]:
logfat_3 = logfat_3.rename(columns={"isocode": "iso3"})
logfat_3 = logfat_3.rename(columns={"int_lnbest_03_all": "logfat_risk_3"})
logfat_3 = logfat_3.rename(columns={"period": "month"})

# Convert period to datetime 
logfat_3["month"] = pd.to_datetime(logfat_3["month"], format="%Y%m")

logfat_12 = logfat_12.rename(columns={"isocode": "iso3"})
logfat_12 = logfat_12.rename(columns={"int_lnbest_12_all": "logfat_risk_12"})
logfat_12 = logfat_12.rename(columns={"period": "month"})

# Convert period to datetime 
logfat_12["month"] = pd.to_datetime(logfat_12["month"], format="%Y%m")

logfat_3 = logfat_3.sort_values(["iso3", "month"])
logfat_12 = logfat_12.sort_values(["iso3", "month"])

cols_keep_3 = [
    "iso3",
    "month",
    "logfat_risk_3"
]

cols_to_keep_12 = [
    "iso3",
    "month",
    "logfat_risk_12"
]

logfat_3 = logfat_3[cols_keep_3].copy()
logfat_12 = logfat_12[cols_to_keep_12].copy()

Delete the rows that have missing values

In [ ]:
logfat_3 = logfat_3.dropna().reset_index(drop=True)
logfat_12 = logfat_12.dropna().reset_index(drop=True)

Save all of them as a file together. First of all we are going to check that the four dataframes contain information of the same countries and months:

In [67]:
def get_countries(df, nom_df):
    if 'iso3' in df.index.names:
        return set(df.index.get_level_values('iso3').unique())
    elif 'iso3' in df.columns:
        return set(df['iso3'].unique())
    else:
        print(f"Column 'iso3' not found in {nom_df}")
        return set()

# Get countries for each DataFrame
countries_r3 = get_countries(risk_3, "risk_3")
countries_r12 = get_countries(risk_12, "risk_12")
countries_l3 = get_countries(logfat_3, "logfat_3")
countries_l12 = get_countries(logfat_12, "logfat_12")

# Check if they are identical
equal = (countries_r3 == countries_r12 == countries_l3 == countries_l12)

print("="*60)
print(f"The four DataFrames have the same countries: {'Yes' if equal else 'No'}")
print("="*60)

# If not equal, show details
if not equal:
    # Common countries in all 4 DataFrames
    common = countries_r3.intersection(countries_r12, countries_l3, countries_l12)
    # All unique countries that appear at least once
    all_detected = countries_r3.union(countries_r12, countries_l3, countries_l12)
    
    print(f"Common countries in all 4 ({len(common)}): {sorted(list(common))}\n")
    
    print("Details:")
    
    # Differences with risk_12
    if countries_r3 != countries_r12:
        if countries_r12 - countries_r3: print(f"  • In 'risk_12' left over: {countries_r12 - countries_r3}")
        if countries_r3 - countries_r12: print(f"  • In 'risk_12' missing: {countries_r3 - countries_r12}")
        
    # Differences with logfat_3
    if countries_r3 != countries_l3:
        if countries_l3 - countries_r3: print(f"  • In 'logfat_3' left over: {countries_l3 - countries_r3}")
        if countries_r3 - countries_l3: print(f"  • In 'logfat_3' missing: {countries_r3 - countries_l3}")
        
    # Differences with logfat_12
    if countries_r3 != countries_l12:
        if countries_l12 - countries_r3: print(f"  • In 'logfat_12' left over: {countries_l12 - countries_r3}")
        if countries_r3 - countries_l12: print(f"  • In 'logfat_12' missing: {countries_r3 - countries_l12}")
else:
    print(f"Detected countries ({len(countries_r3)}): {sorted(list(countries_r3))}")

The four DataFrames have the same countries: Yes
Detected countries (182): ['AFG', 'AGO', 'ALB', 'ARE', 'ARG', 'ARM', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BFA', 'BGD', 'BGR', 'BHR', 'BHS', 'BIH', 'BLR', 'BLZ', 'BMU', 'BOL', 'BRA', 'BRB', 'BRN', 'BTN', 'BWA', 'CAF', 'CAN', 'CHE', 'CHL', 'CHN', 'CIV', 'CMR', 'COD', 'COL', 'COM', 'CRI', 'CUB', 'CYP', 'CZE', 'DEU', 'DJI', 'DNK', 'DOM', 'DZA', 'ECU', 'EGY', 'ERI', 'ESP', 'EST', 'ETH', 'FIN', 'FJI', 'FRA', 'GAB', 'GBR', 'GEO', 'GHA', 'GIN', 'GMB', 'GNB', 'GNQ', 'GRC', 'GRD', 'GTM', 'GUY', 'HKG', 'HND', 'HRV', 'HTI', 'HUN', 'IDN', 'IND', 'IRL', 'IRN', 'IRQ', 'ISL', 'ISR', 'ITA', 'JAM', 'JOR', 'JPN', 'KAZ', 'KEN', 'KGZ', 'KHM', 'KOR', 'KWT', 'LAO', 'LBN', 'LBR', 'LBY', 'LKA', 'LSO', 'LTU', 'LUX', 'LVA', 'MAC', 'MAR', 'MDA', 'MDG', 'MDV', 'MEX', 'MKD', 'MLI', 'MLT', 'MMR', 'MNE', 'MNG', 'MOZ', 'MRT', 'MUS', 'MWI', 'MYS', 'NAM', 'NER', 'NGA', 'NIC', 'NLD', 'NOR', 'NPL', 'NZL', 'OMN', 'PAK', 'PAN', 'PER', 'PHL', 'PNG', 'POL', 'PRI', 'PRK', 

In [ ]:
econAI = (
    risk_3
    .merge(risk_12, on=["iso3", "month"])
    .merge(logfat_3, on=["iso3", "month"])
    .merge(logfat_12, on=["iso3", "month"])
)

In [71]:
econAI.head()

,iso3,month,risk_3,risk_12,logfat_risk_3,logfat_risk_12
0,AFG,2010-01-01,0.983878,0.987250,6.844159,8.102876
1,AFG,2010-02-01,0.991798,0.988044,6.883800,8.129857
2,AFG,2010-03-01,0.995489,0.989177,6.898818,8.131346
3,AFG,2010-04-01,0.999792,0.992189,6.939510,8.149257
4,AFG,2010-05-01,0.998218,0.992714,6.938101,8.141677


In [72]:
econAI.to_csv("../data_clean/econAI_clean.csv", index=False)

### Google Trends

Now we are going to use Google Trends related to displacement to create a signal of a future wave of displacements.

These are the countries for which we are doing our study:

['AB9', 'AFG', 'AGO', 'ARM', 'AUS', 'AZE', 'BDI', 'BEN', 'BFA', 'BGD', 'BHR',
 'BIH', 'BLR', 'BOL', 'BRA', 'CAF', 'CIV', 'CMR', 'COD', 'COL', 'COM', 'CYP',
 'DJI', 'ECU', 'EGY', 'ETH', 'FRA', 'GBR', 'GHA', 'GIN', 'GMB', 'GRC', 'HND',
 'HTI', 'IDN', 'IND', 'IRN', 'IRQ', 'ISR', 'ITA', 'KAZ', 'KEN', 'KGZ', 'KHM',
 'LBN', 'LBR', 'LBY', 'LKA', 'MDG', 'MEX', 'MLI', 'MMR', 'MOZ', 'MWI', 'MYT',
 'NCL', 'NER', 'NGA', 'NIC', 'NLD', 'PAK', 'PER', 'PHL', 'PNG', 'PSE', 'QAT',
 'ROU', 'RUS', 'SDN', 'SLB', 'SLE', 'SLV', 'SOM', 'SSD', 'SYR', 'TCD', 'TGO',
 'THA', 'TJK', 'TUR', 'TZA', 'UGA', 'UKR', 'USA', 'VEN', 'YEM', 'ZAF', 'ZMB',
 'ZWE']

We don't put 'AB9' because is a special territory.

['AF','AO','AM','AU','AZ','BI','BJ','BF','BD','BH','BA','BY','BO','BR','CF','CI','CM','CD','CO','KM','CY','DJ','EC','EG','ET','FR','GB','GH','GN','GM','GR','HN','HT','ID','IN','IR','IQ','IL','IT','KZ','KE','KG','KH','LB','LR','LY','LK','MG','MX','ML','MM','MZ','MW',
'YT','NC','NE','NG','NI','NL','PK','PE','PH','PG','PS','QA','RO','RU','SD','SB','SL','SV','SO','SS','SY','TD','TG','TH','TJ','TR','TZ','UG','UA','US','VE','YE','ZA','ZM','ZW']

88 countries:

countries_names = [
    "Abyei Area",  # AB9 (special territory)
    "Afghanistan",
    "Angola",
    "Armenia",
    "Australia",
    "Azerbaijan",
    "Burundi",
    "Benin",
    "Burkina Faso",
    "Bangladesh",
    "Bahrain",
    "Bosnia and Herzegovina",
    "Belarus",
    "Bolivia",
    "Brazil",
    "Central African Republic",
    "Côte d'Ivoire",
    "Cameroon",
    "Democratic Republic of the Congo",
    "Colombia",
    "Comoros",
    "Cyprus",
    "Djibouti",
    "Ecuador",
    "Egypt",
    "Ethiopia",
    "France",
    "United Kingdom",
    "Ghana",
    "Guinea",
    "Gambia",
    "Greece",
    "Honduras",
    "Haiti",
    "Indonesia",
    "India",
    "Iran",
    "Iraq",
    "Israel",
    "Italy",
    "Kazakhstan",
    "Kenya",
    "Kyrgyzstan",
    "Cambodia",
    "Lebanon",
    "Liberia",
    "Libya",
    "Sri Lanka",
    "Madagascar",
    "Mexico",
    "Mali",
    "Myanmar",
    "Mozambique",
    "Malawi",
    "Mayotte",
    "New Caledonia",
    "Niger",
    "Nigeria",
    "Nicaragua",
    "Netherlands",
    "Pakistan",
    "Peru",
    "Philippines",
    "Papua New Guinea",
    "Palestine",
    "Qatar",
    "Romania",
    "Russia",
    "Sudan",
    "Solomon Islands",
    "Sierra Leone",
    "El Salvador",
    "Somalia",
    "South Sudan",
    "Syria",
    "Chad",
    "Togo",
    "Thailand",
    "Tajikistan",
    "Turkey",
    "Tanzania",
    "Uganda",
    "Ukraine",
    "United States",
    "Venezuela",
    "Yemen",
    "South Africa",
    "Zambia",
    "Zimbabwe"
]

We will get the data loading the google search for each country at month level. And we will save the data in 89 different files.

In [73]:
# Mapping dictionary for iso2 to iso3 (for the 88 countries in our dataset)
iso3_list = ['AFG', 'AGO', 'ARM', 'AUS', 'AZE', 'BDI', 'BEN', 'BFA', 'BGD', 'BHR', 'BIH', 'BLR', 'BOL', 'BRA', 'CAF', 'CIV', 'CMR', 'COD', 'COL', 'COM', 'CYP', 'DJI', 'ECU', 'EGY', 'ETH', 'FRA', 'GBR', 'GHA', 'GIN', 'GMB', 'GRC', 'HND', 'HTI', 'IDN', 'IND', 'IRN', 'IRQ', 'ISR', 'ITA', 'KAZ', 'KEN', 'KGZ', 'KHM', 'LBN', 'LBR', 'LBY', 'LKA', 'MDG', 'MEX', 'MLI', 'MMR', 'MOZ', 'MWI', 'MYT', 'NCL', 'NER', 'NGA', 'NIC', 'NLD', 'PAK', 'PER', 'PHL', 'PNG', 'PSE', 'QAT', 'ROU', 'RUS', 'SDN', 'SLB', 'SLE', 'SLV', 'SOM', 'SSD', 'SYR', 'TCD', 'TGO', 'THA', 'TJK', 'TUR', 'TZA', 'UGA', 'UKR', 'USA', 'VEN', 'YEM', 'ZAF', 'ZMB', 'ZWE']
iso2_list = ['AF', 'AO', 'AM', 'AU', 'AZ', 'BI', 'BJ', 'BF', 'BD', 'BH', 'BA', 'BY', 'BO', 'BR', 'CF', 'CI', 'CM', 'CD', 'CO', 'KM', 'CY', 'DJ', 'EC', 'EG', 'ET', 'FR', 'GB', 'GH', 'GN', 'GM', 'GR', 'HN', 'HT', 'ID', 'IN', 'IR', 'IQ', 'IL', 'IT', 'KZ', 'KE', 'KG', 'KH', 'LB', 'LR', 'LY', 'LK', 'MG', 'MX', 'ML', 'MM', 'MZ', 'MW', 'YT', 'NC', 'NE', 'NG', 'NI', 'NL', 'PK', 'PE', 'PH', 'PG', 'PS', 'QA', 'RO', 'RU', 'SD', 'SB', 'SL', 'SV', 'SO', 'SS', 'SY', 'TD', 'TG', 'TH', 'TJ', 'TR', 'TZ', 'UG', 'UA', 'US', 'VE', 'YE', 'ZA', 'ZM', 'ZW']
mapping = dict(zip(iso2_list, iso3_list))

all_dataframes = []

folder_path = '../data_raw/Google Trends' 

for iso2 in iso2_list:
    file_name = f"{iso2}.csv"
    file_path = os.path.join(folder_path, file_name)
    
    if os.path.exists(file_path):
        # Load the data
        df = pd.read_csv(file_path)
        
        # Create the month and iso3 columns
        df['month'] = pd.to_datetime(df['Time'])
        df['iso3'] = mapping[iso2]
        
        # Sort the columns
        cols_to_keep = ['month', 'iso3', 'Flight', 'Airport', 'Travel', 'Train', 'Bus', 'Passport', 'Travel visa', 'Right of asylum']
        df = df[cols_to_keep]
        
        all_dataframes.append(df)
    else:
        print(f"Warning: File not found for {file_name}")

# Concatenate all dataframes into one
google = pd.concat(all_dataframes, ignore_index=True)

print("-" * 30)
print(f"Done! {len(all_dataframes)} files processed.")

------------------------------
Done! 88 files processed.


In [74]:
# Save the final dataframe to a new CSV file
google.to_csv('../data_clean/google_trends_clean.csv', index=False)

### INFORM INDEX

In [75]:
## Descargar Panel principal 2016-2025

import requests

# workflows válidos
workflow_mapping = {
    2018: 360,
    2019: 370,
    2020: 386,
    2021: 419,
    2022: 433,
    2023: 453,
    2024: 469,
    2025: 482
}

# indicadores seleccionados
selected_indicators = ["INFORM", "VU", "CC", "HA"]

all_years = []

for year, workflow_id in workflow_mapping.items():

    print(f"Processing year {year}...")

    url = f"https://drmkc.jrc.ec.europa.eu/inform-index/API/InformAPI/Countries/Scores/?WorkflowId={workflow_id}"

    data = requests.get(url).json()

    temp_df = pd.DataFrame(data)

    # mantener sólo indicadores seleccionados
    temp_df = temp_df[
        temp_df["IndicatorId"].isin(selected_indicators)
    ]

    # pivot
    temp_df = temp_df.pivot(
        index="Iso3",
        columns="IndicatorId",
        values="IndicatorScore"
    ).reset_index()

    # agregar año
    temp_df["Year"] = year

    # reordenar columnas
    temp_df = temp_df[
        ["Iso3", "Year", "INFORM", "VU", "CC", "HA"]
    ]

    all_years.append(temp_df)

# combinar todos los años
inform = pd.concat(all_years, ignore_index=True)

print(inform.head())

print(inform.shape)

Processing year 2018...
Processing year 2019...
Processing year 2020...
Processing year 2021...
Processing year 2022...
Processing year 2023...
Processing year 2024...
Processing year 2025...
IndicatorId Iso3  Year  INFORM   VU   CC   HA
0            AFG  2018     7.7  7.1  7.5  8.7
1            AGO  2018     5.2  4.6  7.3  4.3
2            ALB  2018     2.7  1.5  4.2  3.3
3            ARE  2018     2.0  1.2  1.9  3.7
4            ARG  2018     2.3  1.4  3.7  2.5
(1528, 6)


In [76]:
import pandas as pd

# 1. Rename column to lowercase
inform = inform.rename(columns={"Iso3": "iso3"})

# 2. Convert Year to a datetime object (set to January 1st of each year)
inform['month'] = pd.to_datetime(inform['Year'].astype(str) + "-01-01")

# 3. Create a master timeline
# We generate a full range of months from the first year (2018) to April 2026
date_range = pd.date_range(start="2018-01-01", end="2026-04-01", freq="MS")

# 4. Expand the grid: Every country x every month
countries = inform['iso3'].unique()
multi_index = pd.MultiIndex.from_product(
    [countries, date_range], 
    names=['iso3', 'month']
)
expanded_df = pd.DataFrame(index=multi_index).reset_index()

# 5. Merge the annual data into the monthly grid
# Annual values will land on January 1st; other months will be NaN initially
inform_monthly = pd.merge(
    expanded_df, 
    inform.drop(columns=["Year"]), 
    on=["iso3", "month"], 
    how="left"
)

# 6. Forward Fill (ffill) the data
# We group by country to ensure values stay within the correct geographical borders
inform_monthly = inform_monthly.sort_values(["iso3", "month"])
indicator_cols = ["INFORM", "VU", "CC", "HA"]
inform_monthly[indicator_cols] = inform_monthly.groupby("iso3")[indicator_cols].ffill()

# 7. Final Cleanup
# Drop rows where no data was ever found (e.g., if a country has no records at all)
inform_monthly = inform_monthly.dropna(subset=["INFORM"]).reset_index(drop=True)

print(inform_monthly.head(13))  # Showing 13 rows to see the transition from one year to the next
print(f"Final dataset shape: {inform_monthly.shape}")

   iso3      month  INFORM   VU   CC   HA
0   AFG 2018-01-01     7.7  7.1  7.5  8.7
1   AFG 2018-02-01     7.7  7.1  7.5  8.7
2   AFG 2018-03-01     7.7  7.1  7.5  8.7
3   AFG 2018-04-01     7.7  7.1  7.5  8.7
4   AFG 2018-05-01     7.7  7.1  7.5  8.7
5   AFG 2018-06-01     7.7  7.1  7.5  8.7
6   AFG 2018-07-01     7.7  7.1  7.5  8.7
7   AFG 2018-08-01     7.7  7.1  7.5  8.7
8   AFG 2018-09-01     7.7  7.1  7.5  8.7
9   AFG 2018-10-01     7.7  7.1  7.5  8.7
10  AFG 2018-11-01     7.7  7.1  7.5  8.7
11  AFG 2018-12-01     7.7  7.1  7.5  8.7
12  AFG 2019-01-01     7.8  7.2  7.5  8.8
Final dataset shape: (19100, 6)


In [77]:
# guardar panel principal
inform_monthly.to_csv("../data_clean/inform_clean.csv", index=False)